# Week 20, Day 4: KL divergence and cross-entropy

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Compute the divergence

Write `kl(p, q)` in bits (sum over entries where p is positive; return infinity if q is zero there) and `cross_entropy(p, q)` = −Σ p log₂ q. Check the asymmetry on the given p and q.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1951)
def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]
    return -np.sum(p * np.log2(p))

def kl(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    m = p > 0
    if np.any(q[m] == 0):
        return np.inf
    return ...

def cross_entropy(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    m = p > 0
    return ...

p = np.array([0.7, 0.2, 0.05, 0.05])
q = np.array([0.25, 0.25, 0.25, 0.25])
print(f"KL(p||q) = {kl(p, q):.3f}, KL(q||p) = {kl(q, p):.3f} bits;  H(p) = {entropy(p):.3f}, H(p, q) = {cross_entropy(p, q):.3f}")

In [ ]:
import numpy as np
def test_zero():
    "KL of a distribution from itself is 0"
    assert abs(kl([0.2, 0.3, 0.5], [0.2, 0.3, 0.5])) < 1e-12

def test_asym():
    "the two directions differ for p and q"
    assert abs(kl(p, q) - kl(q, p)) > 0.1

def test_inf():
    "KL is infinite when q is zero where p is not"
    assert np.isinf(kl([0.5, 0.5], [1.0, 0.0]))

def test_ce():
    "cross-entropy equals entropy plus KL"
    assert abs(cross_entropy(p, q) - (entropy(p) + kl(p, q))) < 1e-12

test_zero()
test_asym()
test_inf()
test_ce()
print("All checks passed")

### Exercise: The Gaussian formula, checked

Write `kl_gauss(m1, s1, m2, s2)` in nats. Check KL(N(0, 1) ‖ N(1, 2²)) against a Monte Carlo estimate `kl_mc`: the mean of log p(x) − log q(x) over 500,000 samples from p.

In [ ]:
def kl_gauss(m1, s1, m2, s2):
    return ...

def log_norm(x, m, s): return -0.5 * np.log(2 * np.pi * s ** 2) - (x - m) ** 2 / (2 * s ** 2)
x = rng.normal(0, 1, 500000)
kl_mc = ...
print(f"formula {kl_gauss(0, 1, 1, 2):.4f} nats, Monte Carlo {kl_mc:.4f};  reverse direction {kl_gauss(1, 2, 0, 1):.4f}")

In [ ]:
import numpy as np
def test_formula():
    "kl_gauss is zero for identical Gaussians and matches the Monte Carlo estimate to within 0.01 nats"
    assert abs(kl_gauss(1, 2, 1, 2)) < 1e-12 and abs(kl_gauss(0, 1, 1, 2) - kl_mc) < 0.01

def test_asym():
    "the reverse direction differs"
    assert abs(kl_gauss(0, 1, 1, 2) - kl_gauss(1, 2, 0, 1)) > 0.3

test_formula()
test_asym()
print("All checks passed")

### Exercise: Predicting the next bin

A refractory neuron fires at about 80 Hz in 2 ms bins. Fit two predictors on the first half of the train: a rate model (P(spike) = overall rate) and a context model (P(spike | previous two bins), estimated by counting, with one pseudo-count each for spike and no spike). Compute each one's cross-entropy in bits per bin on the second half, storing the results in `ce['rate']` and `ce['context']`.

In [ ]:
T = 400000
p0 = 80 * 0.002; q_r = p0 / (1 - 2 * p0)
train = np.zeros(T, int); t = 0
while t < T:
    if rng.random() < q_r:
        train[t] = 1; t += 3
    else:
        t += 1
half = T // 2
fit, test = train[:half], train[half:]
p_spike = fit.mean()

def context_index(x):                       # previous two bins as 0..3, for bins 2 onwards
    return 2 * x[1:-1] + x[:-2]

ctx_fit, nxt_fit = context_index(fit), fit[2:]
p_ctx = np.array([(nxt_fit[ctx_fit == c].sum() + 1) / ((ctx_fit == c).sum() + 2) for c in range(4)])

ctx_test, nxt_test = context_index(test), test[2:]
q_rate = np.full(len(nxt_test), p_spike)
q_ctx = ...                                 # predicted spike probability for each test bin
def bits(qs, y): return -np.mean(y * np.log2(qs) + (1 - y) * np.log2(1 - qs))
ce = {"rate": bits(q_rate, nxt_test), "context": ...}
print(f"cross-entropy (bits per bin): rate model {ce['rate']:.4f}, context model {ce['context']:.4f};  P(spike | context) = {p_ctx.round(3)}")

In [ ]:
import numpy as np
def test_rate_model():
    "the rate-only model's cross-entropy equals the binary entropy of the firing probability (within 0.01)"
    assert abs(ce["rate"] - entropy([p_spike, 1 - p_spike])) < 0.01

def test_context_better():
    "the context model has lower test cross-entropy than the rate model"
    assert ce["context"] < ce["rate"] - 0.05

test_rate_model()
test_context_better()
print("All checks passed")